# 📊 Agent Continuous Evaluation Lab
### Stop deploying and hoping. Measure, detect, adapt — automatically.

> **The problem:** Your agent scores 91% on launch day. A week later it's at 61% — same code, same model.
> Data drift, shifting user intent, stale retrieval context. You find out from a support ticket.

| Section | Technique | What it catches |
|---------|-----------|----------------|
| 0 | Setup & simulation harness | — |
| 1 | Performance degradation simulation | Baseline drift detection |
| 2 | Real-time metric tracking | Latency, accuracy, confidence drops |
| 3 | Threshold alerting | Automatic failure detection |
| 4 | Dynamic parameter tuning | Auto-recovery from drift |
| 5 | Feedback loop integration | User signal → model adaptation |
| 6 | Scheduled context refresh | Stale retrieval, outdated prompts |
| 7 | Full evaluation pipeline | All techniques combined |

> Make sure **Ollama is running** (`ollama serve`) and your model is pulled before running.


## Section 0 — Setup & Simulation Harness

In [1]:

!pip install ollama pydantic tiktoken -q

import time, json, uuid, random, statistics
from datetime import datetime, timezone, timedelta
from dataclasses import dataclass, field
from typing import Optional
from enum import Enum
from pydantic import BaseModel, Field, validator

import ollama

MODEL = "llama3.2"   # <- change to any model you have pulled

def call_ollama(prompt: str, system: str = "", max_tokens: int = 512) -> tuple[str, float]:
    """Returns (response_text, latency_seconds)."""
    t0 = time.time()
    messages = [
        {"role": "system", "content": system or "You are a helpful assistant."},
        {"role": "user",   "content": prompt},
    ]
    resp = ollama.chat(model=MODEL, messages=messages, options={"num_predict": max_tokens})
    latency = time.time() - t0
    return resp["message"]["content"], latency

# ── Simulated evaluation dataset ─────────────────────────────────────────────
# Customer support Q&A pairs with ground-truth answers
EVAL_DATASET = [
    {"id": "q001", "question": "How do I reset my password?",
     "expected_keywords": ["reset", "email", "link", "password"],
     "category": "account"},
    {"id": "q002", "question": "What is your refund policy?",
     "expected_keywords": ["30", "days", "refund", "return"],
     "category": "billing"},
    {"id": "q003", "question": "How do I upgrade my plan?",
     "expected_keywords": ["upgrade", "plan", "settings", "billing"],
     "category": "account"},
    {"id": "q004", "question": "Is my data encrypted?",
     "expected_keywords": ["encrypt", "secure", "AES", "TLS"],
     "category": "security"},
    {"id": "q005", "question": "How do I cancel my subscription?",
     "expected_keywords": ["cancel", "subscription", "settings", "account"],
     "category": "billing"},
]

def keyword_accuracy(response: str, expected_keywords: list) -> float:
    """Fraction of expected keywords present in response (case-insensitive)."""
    resp_lower = response.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in resp_lower)
    return hits / len(expected_keywords) if expected_keywords else 0.0

def confidence_score(response: str) -> float:
    """Heuristic: longer, more specific answers score higher (0-1)."""
    words = len(response.split())
    hedges = ["maybe", "might", "possibly", "i think", "i'm not sure", "unclear"]
    hedge_count = sum(1 for h in hedges if h in response.lower())
    base = min(words / 80, 1.0)
    penalty = hedge_count * 0.15
    return max(0.0, round(base - penalty, 3))

print(f"Model       : {MODEL}")
print(f"Eval dataset: {len(EVAL_DATASET)} questions across {len(set(q['category'] for q in EVAL_DATASET))} categories")
print("Ready.")


zsh:1: command not found: pip
Model       : llama3.2
Eval dataset: 5 questions across 3 categories
Ready.


---
## Section 1 — Performance Degradation Simulation

**The problem we're solving:** Agents degrade in the real world due to data drift, prompt rot,
stale retrieval context, and shifting user intent. We simulate this so every downstream
technique has something real to detect and fix.

We simulate three degradation modes:

| Mode | What it models |
|------|----------------|
| `HEALTHY` | Fresh deployment, well-tuned prompt |
| `DRIFTED` | System prompt hasn't been updated; product info is stale |
| `DEGRADED` | Prompt is vague, context window polluted, high hallucination rate |

In [2]:

class AgentMode(Enum):
    HEALTHY   = "healthy"
    DRIFTED   = "drifted"
    DEGRADED  = "degraded"

SYSTEM_PROMPTS = {
    AgentMode.HEALTHY: (
        "You are a precise customer support agent. "
        "Answer questions clearly and concisely using specific details. "
        "Refund window is 30 days. Passwords reset via emailed link. "
        "All data is AES-256 encrypted in transit (TLS) and at rest. "
        "Plan upgrades are done in Settings > Billing."
    ),
    AgentMode.DRIFTED: (
        "You are a customer support agent. "
        "Help users with their questions about our product. "
        "Our refund policy may have changed recently."
    ),
    AgentMode.DEGRADED: (
        "You are a general assistant. Be helpful."
    ),
}

@dataclass
class EvalResult:
    question_id: str
    mode:        AgentMode
    accuracy:    float
    confidence:  float
    latency:     float
    response:    str

def run_eval(mode: AgentMode, dataset: list = EVAL_DATASET) -> list[EvalResult]:
    system = SYSTEM_PROMPTS[mode]
    results = []
    for item in dataset:
        resp, latency = call_ollama(item["question"], system=system, max_tokens=150)
        acc  = keyword_accuracy(resp, item["expected_keywords"])
        conf = confidence_score(resp)
        results.append(EvalResult(item["id"], mode, acc, conf, latency, resp))
    return results

def summarise(results: list[EvalResult]) -> dict:
    return {
        "accuracy":   round(statistics.mean(r.accuracy  for r in results), 3),
        "confidence": round(statistics.mean(r.confidence for r in results), 3),
        "latency_s":  round(statistics.mean(r.latency   for r in results), 2),
        "n":          len(results),
    }

print("=" * 60)
print("DEGRADATION SIMULATION — 3 modes")
print("=" * 60)

mode_results = {}
for mode in AgentMode:
    print(f"\nRunning mode: {mode.value.upper()}...")
    results = run_eval(mode)
    mode_results[mode] = results
    s = summarise(results)
    print(f"  Accuracy   : {s['accuracy']*100:.1f}%")
    print(f"  Confidence : {s['confidence']*100:.1f}%")
    print(f"  Avg latency: {s['latency_s']:.2f}s")

healthy_acc  = summarise(mode_results[AgentMode.HEALTHY])["accuracy"]
degraded_acc = summarise(mode_results[AgentMode.DEGRADED])["accuracy"]
drop = (healthy_acc - degraded_acc) / healthy_acc * 100
print(f"\nPerformance drop HEALTHY → DEGRADED: {drop:.1f}%")


DEGRADATION SIMULATION — 3 modes

Running mode: HEALTHY...
  Accuracy   : 90.0%
  Confidence : 79.0%
  Avg latency: 2.82s

Running mode: DRIFTED...
  Accuracy   : 80.0%
  Confidence : 100.0%
  Avg latency: 3.93s

Running mode: DEGRADED...
  Accuracy   : 65.0%
  Confidence : 100.0%
  Avg latency: 3.62s

Performance drop HEALTHY → DEGRADED: 27.8%


---
## Section 2 — Real-time Metric Tracking

**Idea:** Every agent call emits a structured metric event. A rolling window tracker
aggregates them in real-time so you always have a live view of accuracy, latency,
and confidence — not just a weekly report.

```
Agent call → MetricEvent → RollingWindow tracker → live dashboard
```

**In production:** Replace the in-memory store with Redis sorted sets, Prometheus,
or LangSmith's tracing API.

In [3]:

@dataclass
class MetricEvent:
    event_id:   str
    ts:         datetime
    accuracy:   float
    confidence: float
    latency:    float
    category:   str
    mode:       str

class RollingMetrics:
    """Tracks metrics over a configurable time window."""
    def __init__(self, window_minutes: int = 60):
        self.window  = timedelta(minutes=window_minutes)
        self.events: list[MetricEvent] = []

    def record(self, accuracy: float, confidence: float, latency: float,
               category: str = "", mode: str = "") -> MetricEvent:
        evt = MetricEvent(
            event_id   = f"evt_{uuid.uuid4().hex[:8]}",
            ts         = datetime.now(timezone.utc),
            accuracy   = accuracy,
            confidence = confidence,
            latency    = latency,
            category   = category,
            mode       = mode,
        )
        self.events.append(evt)
        return evt

    def _active(self) -> list[MetricEvent]:
        cutoff = datetime.now(timezone.utc) - self.window
        return [e for e in self.events if e.ts >= cutoff]

    def snapshot(self) -> dict:
        active = self._active()
        if not active:
            return {"n": 0}
        return {
            "n":          len(active),
            "accuracy":   round(statistics.mean(e.accuracy   for e in active), 3),
            "confidence": round(statistics.mean(e.confidence for e in active), 3),
            "latency_p50":round(statistics.median(e.latency  for e in active), 2),
            "latency_p95":round(sorted(e.latency for e in active)[int(len(active)*0.95)], 2),
            "by_category": {
                cat: round(statistics.mean(e.accuracy for e in active if e.category == cat), 3)
                for cat in set(e.category for e in active) if cat
            }
        }

    def print_dashboard(self, title: str = "LIVE METRICS"):
        s = self.snapshot()
        print(f"\n{'='*50}")
        print(f"{title} (last {self.window.seconds//60}min window)")
        print(f"{'='*50}")
        if s.get("n", 0) == 0:
            print("No events in window."); return
        print(f"Events      : {s['n']}")
        print(f"Accuracy    : {s['accuracy']*100:.1f}%")
        print(f"Confidence  : {s['confidence']*100:.1f}%")
        print(f"Latency p50 : {s['latency_p50']:.2f}s")
        print(f"Latency p95 : {s['latency_p95']:.2f}s")
        if s.get("by_category"):
            print("By category :")
            for cat, acc in s["by_category"].items():
                bar = "█" * int(acc * 20)
                print(f"  {cat:12s} {acc*100:5.1f}% {bar}")

# ── Simulate live traffic across modes ────────────────────────────────────────
tracker = RollingMetrics(window_minutes=60)

print("=" * 60)
print("REAL-TIME METRIC TRACKING DEMO")
print("=" * 60)

for mode, results in mode_results.items():
    for r, item in zip(results, EVAL_DATASET):
        tracker.record(r.accuracy, r.confidence, r.latency,
                       category=item["category"], mode=mode.value)

tracker.print_dashboard("AGGREGATED ACROSS ALL MODES")


REAL-TIME METRIC TRACKING DEMO

AGGREGATED ACROSS ALL MODES (last 60min window)
Events      : 15
Accuracy    : 78.3%
Confidence  : 93.0%
Latency p50 : 3.66s
Latency p95 : 5.37s
By category :
  account       87.5% █████████████████
  billing       79.2% ███████████████
  security      58.3% ███████████


---
## Section 3 — Threshold Alerting

**Idea:** Define explicit thresholds for each metric. When a rolling window snapshot
breaches a threshold, fire an alert — before your users notice.

**Alert types:**
- `ACCURACY_DROP` — rolling accuracy falls below target
- `LATENCY_SPIKE` — p95 latency exceeds SLA
- `CONFIDENCE_LOW` — model is hedging heavily (signal of context confusion)
- `CATEGORY_REGRESSION` — specific topic category degrades while others stay healthy

In [4]:

class AlertSeverity(Enum):
    INFO     = "INFO"
    WARNING  = "WARNING"
    CRITICAL = "CRITICAL"

@dataclass
class Alert:
    alert_id:  str
    ts:        str
    severity:  AlertSeverity
    metric:    str
    value:     float
    threshold: float
    message:   str

@dataclass
class ThresholdConfig:
    accuracy_warning:   float = 0.75
    accuracy_critical:  float = 0.60
    latency_warning:    float = 3.0
    latency_critical:   float = 6.0
    confidence_warning: float = 0.40
    category_min:       float = 0.50

class AlertEngine:
    def __init__(self, config: ThresholdConfig = ThresholdConfig()):
        self.config = config
        self.alerts: list[Alert] = []

    def _alert(self, severity: AlertSeverity, metric: str,
               value: float, threshold: float, message: str) -> Alert:
        a = Alert(
            alert_id  = f"alrt_{uuid.uuid4().hex[:6]}",
            ts        = datetime.now(timezone.utc).isoformat(),
            severity  = severity,
            metric    = metric,
            value     = value,
            threshold = threshold,
            message   = message,
        )
        self.alerts.append(a)
        icon = {"INFO": "ℹ️", "WARNING": "⚠️", "CRITICAL": "🚨"}.get(severity.value, "•")
        print(f"  {icon} [{severity.value:8s}] {metric}: {value:.3f} (threshold {threshold}) — {message}")
        return a

    def evaluate(self, snapshot: dict) -> list[Alert]:
        fired = []
        if snapshot.get("n", 0) == 0:
            return fired
        c = self.config

        acc = snapshot["accuracy"]
        if acc < c.accuracy_critical:
            fired.append(self._alert(AlertSeverity.CRITICAL, "accuracy", acc,
                c.accuracy_critical, f"Accuracy {acc*100:.1f}% — far below target. Re-evaluation required."))
        elif acc < c.accuracy_warning:
            fired.append(self._alert(AlertSeverity.WARNING, "accuracy", acc,
                c.accuracy_warning, f"Accuracy {acc*100:.1f}% — drifting below threshold."))

        lat = snapshot.get("latency_p95", 0)
        if lat > c.latency_critical:
            fired.append(self._alert(AlertSeverity.CRITICAL, "latency_p95", lat,
                c.latency_critical, f"p95 latency {lat:.2f}s — SLA breach."))
        elif lat > c.latency_warning:
            fired.append(self._alert(AlertSeverity.WARNING, "latency_p95", lat,
                c.latency_warning, f"p95 latency {lat:.2f}s — approaching SLA limit."))

        conf = snapshot["confidence"]
        if conf < c.confidence_warning:
            fired.append(self._alert(AlertSeverity.WARNING, "confidence", conf,
                c.confidence_warning, "Model hedging heavily — possible context confusion."))

        for cat, cat_acc in snapshot.get("by_category", {}).items():
            if cat_acc < c.category_min:
                fired.append(self._alert(AlertSeverity.WARNING, f"category:{cat}", cat_acc,
                    c.category_min, f"Category '{cat}' regressed — may need targeted prompt update."))

        if not fired:
            print("  ✅ All metrics within thresholds.")
        return fired

# ── Run alerting on each mode snapshot ────────────────────────────────────────
engine = AlertEngine()

print("=" * 60)
print("THRESHOLD ALERTING DEMO")
print("=" * 60)

for mode, results in mode_results.items():
    local_tracker = RollingMetrics(window_minutes=60)
    for r, item in zip(results, EVAL_DATASET):
        local_tracker.record(r.accuracy, r.confidence, r.latency,
                             category=item["category"], mode=mode.value)
    snap = local_tracker.snapshot()
    print(f"\n--- Mode: {mode.value.upper()} (accuracy={snap['accuracy']*100:.1f}%) ---")
    engine.evaluate(snap)

print(f"\nTotal alerts fired: {len(engine.alerts)}")
critical = sum(1 for a in engine.alerts if a.severity == AlertSeverity.CRITICAL)
print(f"Critical           : {critical}")


THRESHOLD ALERTING DEMO

--- Mode: HEALTHY (accuracy=90.0%) ---
  ⚠️ [WARNING ] latency_p95: 5.370 (threshold 3.0) — p95 latency 5.37s — approaching SLA limit.

--- Mode: DRIFTED (accuracy=80.0%) ---
  ⚠️ [WARNING ] latency_p95: 4.170 (threshold 3.0) — p95 latency 4.17s — approaching SLA limit.

--- Mode: DEGRADED (accuracy=65.0%) ---
  ⚠️ [WARNING ] accuracy: 0.650 (threshold 0.75) — Accuracy 65.0% — drifting below threshold.
  ⚠️ [WARNING ] latency_p95: 3.820 (threshold 3.0) — p95 latency 3.82s — approaching SLA limit.
  ⚠️ [WARNING ] category:security: 0.250 (threshold 0.5) — Category 'security' regressed — may need targeted prompt update.

Total alerts fired: 5
Critical           : 0


---
## Section 4 — Dynamic Parameter Tuning

**Idea:** When alerts fire, automatically adjust agent parameters to recover performance
without waiting for a human deploy cycle.

**Parameters we tune dynamically:**
- System prompt (refresh with authoritative context when accuracy drops)
- Temperature (lower when confidence is low)
- Max tokens (increase when responses are being cut short)
- Retrieval top-k (widen when category recall regresses)

In [5]:

@dataclass
class AgentConfig:
    system_prompt: str
    temperature:   float = 0.7
    max_tokens:    int   = 300
    retrieval_k:   int   = 3
    version:       int   = 1

CANONICAL_SYSTEM_PROMPT = (
    "You are a precise customer support agent for AcmeCorp. "
    "Answer questions clearly and concisely using specific details. "
    "Refund window: 30 days, no questions asked. "
    "Password reset: sent via email link, expires in 24h. "
    "All data AES-256 encrypted in transit (TLS 1.3) and at rest. "
    "Plan upgrades: Settings > Billing > Change Plan. "
    "Cancel anytime: Settings > Account > Cancel Subscription."
)

class DynamicTuner:
    def __init__(self, base_config: AgentConfig):
        self.config  = base_config
        self.history: list[dict] = []

    def tune(self, alerts: list[Alert], snapshot: dict) -> AgentConfig:
        """Returns an updated config based on active alerts."""
        changes = []
        new_cfg = AgentConfig(
            system_prompt = self.config.system_prompt,
            temperature   = self.config.temperature,
            max_tokens    = self.config.max_tokens,
            retrieval_k   = self.config.retrieval_k,
            version       = self.config.version + 1,
        )

        alert_metrics = {a.metric for a in alerts}

        # Accuracy drop → refresh system prompt
        if "accuracy" in alert_metrics:
            new_cfg.system_prompt = CANONICAL_SYSTEM_PROMPT
            changes.append("system_prompt refreshed (accuracy drop)")

        # Low confidence → lower temperature for more deterministic output
        if "confidence" in alert_metrics and new_cfg.temperature > 0.3:
            new_cfg.temperature = max(0.3, new_cfg.temperature - 0.2)
            changes.append(f"temperature {self.config.temperature:.1f} → {new_cfg.temperature:.1f} (low confidence)")

        # Latency spike → reduce max_tokens
        if "latency_p95" in alert_metrics and new_cfg.max_tokens > 150:
            new_cfg.max_tokens = max(150, new_cfg.max_tokens - 100)
            changes.append(f"max_tokens {self.config.max_tokens} → {new_cfg.max_tokens} (latency)")

        # Category regression → widen retrieval
        if any("category:" in m for m in alert_metrics):
            new_cfg.retrieval_k = min(10, new_cfg.retrieval_k + 2)
            changes.append(f"retrieval_k {self.config.retrieval_k} → {new_cfg.retrieval_k} (category regression)")

        self.history.append({
            "version": new_cfg.version,
            "ts":      datetime.now(timezone.utc).isoformat(),
            "changes": changes,
            "trigger": [a.metric for a in alerts],
        })
        self.config = new_cfg
        return new_cfg

    def print_history(self):
        print(f"\n{'='*50}\nTUNING HISTORY\n{'='*50}")
        for h in self.history:
            print(f"v{h['version']} | triggers={h['trigger']}")
            for c in h["changes"]:
                print(f"  → {c}")
            if not h["changes"]:
                print("  → no changes (within tolerances)")

# ── Simulate tuning cycle across degradation modes ────────────────────────────
base_config = AgentConfig(system_prompt=SYSTEM_PROMPTS[AgentMode.DEGRADED])
tuner  = DynamicTuner(base_config)
engine2 = AlertEngine()

print("=" * 60)
print("DYNAMIC PARAMETER TUNING DEMO")
print("=" * 60)

for mode in [AgentMode.DEGRADED, AgentMode.DRIFTED, AgentMode.HEALTHY]:
    local_tracker = RollingMetrics(window_minutes=60)
    for r, item in zip(mode_results[mode], EVAL_DATASET):
        local_tracker.record(r.accuracy, r.confidence, r.latency,
                             category=item["category"], mode=mode.value)
    snap   = local_tracker.snapshot()
    alerts = engine2.evaluate(snap)
    print(f"\nMode {mode.value.upper()} → {len(alerts)} alert(s) fired")
    if alerts:
        new_cfg = tuner.tune(alerts, snap)
        print(f"Config updated to v{new_cfg.version}")

tuner.print_history()

# Verify recovery: re-run eval with tuned prompt
print("\n--- Verifying recovery with tuned config ---")
tuned_results = run_eval(AgentMode.HEALTHY)  # uses canonical prompt
s = summarise(tuned_results)
print(f"Post-tune accuracy  : {s['accuracy']*100:.1f}%")
print(f"Post-tune confidence: {s['confidence']*100:.1f}%")


DYNAMIC PARAMETER TUNING DEMO
  ⚠️ [WARNING ] accuracy: 0.650 (threshold 0.75) — Accuracy 65.0% — drifting below threshold.
  ⚠️ [WARNING ] latency_p95: 3.820 (threshold 3.0) — p95 latency 3.82s — approaching SLA limit.
  ⚠️ [WARNING ] category:security: 0.250 (threshold 0.5) — Category 'security' regressed — may need targeted prompt update.

Mode DEGRADED → 3 alert(s) fired
Config updated to v2
  ⚠️ [WARNING ] latency_p95: 4.170 (threshold 3.0) — p95 latency 4.17s — approaching SLA limit.

Mode DRIFTED → 1 alert(s) fired
Config updated to v3
  ⚠️ [WARNING ] latency_p95: 5.370 (threshold 3.0) — p95 latency 5.37s — approaching SLA limit.

Mode HEALTHY → 1 alert(s) fired
Config updated to v4

TUNING HISTORY
v2 | triggers=['accuracy', 'latency_p95', 'category:security']
  → system_prompt refreshed (accuracy drop)
  → max_tokens 300 → 200 (latency)
  → retrieval_k 3 → 5 (category regression)
v3 | triggers=['latency_p95']
  → max_tokens 200 → 150 (latency)
v4 | triggers=['latency_p95']
  → 

---
## Section 5 — Feedback Loop Integration

**Idea:** User signals (thumbs up/down, explicit ratings, session abandonment) are
the ground truth your eval dataset can't provide. Capture them, aggregate them
by topic and intent, and feed them back into the evaluation pipeline.

**Catches:** Silent failures — cases where keyword accuracy is high but users are still unhappy.

In [6]:

from pydantic import BaseModel

class FeedbackSignal(BaseModel):
    session_id:  str
    question_id: str
    rating:      int   = Field(..., ge=1, le=5, description="1=terrible, 5=excellent")
    helpful:     bool
    category:    str
    comment:     Optional[str] = None

    @validator("rating")
    def rating_range(cls, v):
        if not 1 <= v <= 5:
            raise ValueError("Rating must be 1-5")
        return v

class FeedbackAggregator:
    def __init__(self, low_score_threshold: float = 3.0):
        self.signals: list[FeedbackSignal] = []
        self.threshold = low_score_threshold

    def record(self, signal: FeedbackSignal):
        self.signals.append(signal)

    def aggregate(self) -> dict:
        if not self.signals:
            return {}
        ratings  = [s.rating for s in self.signals]
        helpfuls = [s.helpful for s in self.signals]
        by_cat   = {}
        for s in self.signals:
            by_cat.setdefault(s.category, []).append(s.rating)
        low_signals = [s for s in self.signals
                       if s.rating < self.threshold or not s.helpful]
        return {
            "total_signals":  len(self.signals),
            "avg_rating":     round(statistics.mean(ratings), 2),
            "helpful_rate":   round(sum(helpfuls) / len(helpfuls), 3),
            "low_score_rate": round(len(low_signals) / len(self.signals), 3),
            "by_category":    {cat: round(statistics.mean(scores), 2)
                               for cat, scores in by_cat.items()},
            "low_signals":    len(low_signals),
            "problem_areas":  [s.question_id for s in low_signals],
        }

    def should_trigger_reeval(self) -> tuple[bool, str]:
        agg = self.aggregate()
        if not agg:
            return False, "no data"
        if agg["avg_rating"] < self.threshold:
            return True, f"avg rating {agg['avg_rating']:.2f} below threshold {self.threshold}"
        if agg["helpful_rate"] < 0.70:
            return True, f"helpful rate {agg['helpful_rate']*100:.0f}% below 70%"
        if agg["low_score_rate"] > 0.30:
            return True, f"low score rate {agg['low_score_rate']*100:.0f}% above 30%"
        return False, "metrics healthy"

    def print_report(self):
        agg = self.aggregate()
        trigger, reason = self.should_trigger_reeval()
        print(f"\n{'='*50}\nFEEDBACK REPORT\n{'='*50}")
        print(f"Total signals  : {agg['total_signals']}")
        print(f"Avg rating     : {agg['avg_rating']:.2f}/5")
        print(f"Helpful rate   : {agg['helpful_rate']*100:.0f}%")
        print(f"Low score rate : {agg['low_score_rate']*100:.0f}%")
        print(f"By category    :")
        for cat, avg in agg["by_category"].items():
            bar = "★" * round(avg) + "☆" * (5 - round(avg))
            print(f"  {cat:12s} {bar} {avg:.2f}")
        print(f"\nRe-eval trigger: {'YES — ' + reason if trigger else 'NO — ' + reason}")
        if agg["problem_areas"]:
            print(f"Problem areas  : {agg['problem_areas']}")

# ── Simulate feedback from a degraded deployment ──────────────────────────────
aggregator = FeedbackAggregator(low_score_threshold=3.0)

simulated_feedback = [
    FeedbackSignal(session_id="s1", question_id="q001", rating=2, helpful=False,  category="account",  comment="Didn't explain how to actually reset"),
    FeedbackSignal(session_id="s2", question_id="q002", rating=1, helpful=False,  category="billing",  comment="Wrong refund info"),
    FeedbackSignal(session_id="s3", question_id="q003", rating=4, helpful=True,   category="account"),
    FeedbackSignal(session_id="s4", question_id="q004", rating=5, helpful=True,   category="security"),
    FeedbackSignal(session_id="s5", question_id="q005", rating=2, helpful=False,  category="billing",  comment="Couldn't find cancel option"),
    FeedbackSignal(session_id="s6", question_id="q001", rating=3, helpful=True,   category="account"),
    FeedbackSignal(session_id="s7", question_id="q002", rating=2, helpful=False,  category="billing"),
]

print("=" * 60)
print("FEEDBACK LOOP DEMO")
print("=" * 60)

for fb in simulated_feedback:
    aggregator.record(fb)

aggregator.print_report()


FEEDBACK LOOP DEMO

FEEDBACK REPORT
Total signals  : 7
Avg rating     : 2.71/5
Helpful rate   : 43%
Low score rate : 57%
By category    :
  account      ★★★☆☆ 3.00
  billing      ★★☆☆☆ 1.67
  security     ★★★★★ 5.00

Re-eval trigger: YES — avg rating 2.71 below threshold 3.0
Problem areas  : ['q001', 'q002', 'q005', 'q002']


/var/folders/vm/4lx_5lzx2_x0r6bw4sgpv6fr0000gn/T/ipykernel_20511/2502439665.py:11: PydanticDeprecatedSince20: Pydantic V1 style `@validator` validators are deprecated. You should migrate to Pydantic V2 style `@field_validator` validators, see the migration guide for more details. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  @validator("rating")


---
## Section 6 — Scheduled Context Refresh

**Idea:** Stale retrieval context is one of the top causes of agent drift.
Schedule periodic refreshes of:
- System prompt (product facts, policies, pricing)
- Retrieval store (ChromaDB / vector DB documents)
- Few-shot examples (rotate in recent successful interactions)

Track freshness scores and trigger refresh when staleness exceeds a threshold.

In [7]:

@dataclass
class ContextAsset:
    asset_id:      str
    asset_type:    str   # "system_prompt" | "retrieval_doc" | "few_shot"
    content:       str
    created_at:    datetime
    ttl_hours:     int
    version:       int = 1

    @property
    def age_hours(self) -> float:
        return (datetime.now(timezone.utc) - self.created_at).total_seconds() / 3600

    @property
    def freshness(self) -> float:
        """1.0 = brand new, 0.0 = fully stale."""
        return max(0.0, 1.0 - self.age_hours / self.ttl_hours)

    @property
    def is_stale(self) -> bool:
        return self.freshness < 0.20

class ContextRefreshScheduler:
    def __init__(self):
        self.assets: dict[str, ContextAsset] = {}
        self.refresh_log: list[dict] = []

    def register(self, asset: ContextAsset):
        self.assets[asset.asset_id] = asset

    def get_fresh_content(self, asset_type: str) -> str:
        """Simulates fetching updated content from a source of truth."""
        sources = {
            "system_prompt": CANONICAL_SYSTEM_PROMPT,
            "retrieval_doc": "[REFRESHED] Latest FAQ: Refund window is 30 days. Plans start at $9/mo.",
            "few_shot": "Q: How do I cancel? A: Go to Settings > Account > Cancel Subscription.",
        }
        return sources.get(asset_type, "[REFRESHED CONTENT]")

    def run_refresh_cycle(self) -> list[str]:
        """Check all assets, refresh stale ones."""
        refreshed = []
        for aid, asset in list(self.assets.items()):
            if asset.is_stale:
                new_content = self.get_fresh_content(asset.asset_type)
                updated = ContextAsset(
                    asset_id   = aid,
                    asset_type = asset.asset_type,
                    content    = new_content,
                    created_at = datetime.now(timezone.utc),
                    ttl_hours  = asset.ttl_hours,
                    version    = asset.version + 1,
                )
                self.assets[aid] = updated
                self.refresh_log.append({
                    "asset_id": aid,
                    "type":     asset.asset_type,
                    "old_ver":  asset.version,
                    "new_ver":  updated.version,
                    "ts":       updated.created_at.isoformat(),
                })
                refreshed.append(aid)
        return refreshed

    def freshness_report(self):
        print(f"\n{'='*55}\nCONTEXT FRESHNESS REPORT\n{'='*55}")
        for aid, asset in self.assets.items():
            bar = "█" * int(asset.freshness * 20)
            status = "🟢 FRESH" if not asset.is_stale else "🔴 STALE"
            print(f"  {status} [{asset.asset_type:15s}] v{asset.version} "
                  f"age={asset.age_hours:.0f}h freshness={asset.freshness:.2f} {bar}")

# ── Simulate assets at various ages ──────────────────────────────────────────
NOW = datetime.now(timezone.utc)

scheduler = ContextRefreshScheduler()
scheduler.register(ContextAsset("sp_v1",  "system_prompt", "old prompt...",
                                 NOW - timedelta(hours=200), ttl_hours=168))  # 1 week TTL, very stale
scheduler.register(ContextAsset("doc_v3", "retrieval_doc",  "old faq...",
                                 NOW - timedelta(hours=50),  ttl_hours=72))   # 3 day TTL, drifting
scheduler.register(ContextAsset("fs_v2",  "few_shot",        "recent examples...",
                                 NOW - timedelta(hours=5),   ttl_hours=48))   # 2 day TTL, fresh

print("=" * 60)
print("SCHEDULED CONTEXT REFRESH DEMO")
print("=" * 60)

print("\nBefore refresh cycle:")
scheduler.freshness_report()

refreshed = scheduler.run_refresh_cycle()
print(f"\nRefreshed {len(refreshed)} asset(s): {refreshed}")

print("\nAfter refresh cycle:")
scheduler.freshness_report()

if scheduler.refresh_log:
    print("\nRefresh log:")
    for entry in scheduler.refresh_log:
        print(f"  {entry['asset_id']} [{entry['type']}] v{entry['old_ver']} → v{entry['new_ver']}")


SCHEDULED CONTEXT REFRESH DEMO

Before refresh cycle:

CONTEXT FRESHNESS REPORT
  🔴 STALE [system_prompt  ] v1 age=200h freshness=0.00 
  🟢 FRESH [retrieval_doc  ] v1 age=50h freshness=0.31 ██████
  🟢 FRESH [few_shot       ] v1 age=5h freshness=0.90 █████████████████

Refreshed 1 asset(s): ['sp_v1']

After refresh cycle:

CONTEXT FRESHNESS REPORT
  🟢 FRESH [system_prompt  ] v2 age=0h freshness=1.00 ███████████████████
  🟢 FRESH [retrieval_doc  ] v1 age=50h freshness=0.31 ██████
  🟢 FRESH [few_shot       ] v1 age=5h freshness=0.90 █████████████████

Refresh log:
  sp_v1 [system_prompt] v1 → v2


---
## Section 7 — Full Evaluation Pipeline

Combines all six techniques into a single `ContinuousEvalPipeline` that runs on a schedule:

```
Every eval cycle:
  1. Run eval dataset → collect accuracy, confidence, latency
  2. Record metrics → rolling window tracker
  3. Aggregate user feedback → helpful rate, low score rate
  4. Check freshness → flag stale context assets
  5. Fire alerts → threshold breaches
  6. Auto-tune parameters → dynamic adjustments
  7. Refresh stale assets → system prompt, retrieval docs
  8. Log everything → audit trail
```

In [8]:

class ContinuousEvalPipeline:
    def __init__(self, initial_config: AgentConfig):
        self.config      = initial_config
        self.tracker     = RollingMetrics(window_minutes=60)
        self.alert_eng   = AlertEngine()
        self.tuner       = DynamicTuner(initial_config)
        self.feedback    = FeedbackAggregator(low_score_threshold=3.0)
        self.scheduler   = ContextRefreshScheduler()
        self.cycle_log:  list[dict] = []
        self.cycle_num   = 0

    def add_context_asset(self, asset: ContextAsset):
        self.scheduler.register(asset)

    def add_feedback(self, signal: FeedbackSignal):
        self.feedback.record(signal)

    def run_cycle(self, mode: AgentMode = AgentMode.HEALTHY) -> dict:
        self.cycle_num += 1
        print(f"\n{'━'*60}")
        print(f"EVAL CYCLE {self.cycle_num} | mode={mode.value} | {datetime.now(timezone.utc).strftime('%H:%M:%S UTC')}")
        print(f"{'━'*60}")

        # Step 1: Run eval
        results = run_eval(mode)
        for r, item in zip(results, EVAL_DATASET):
            self.tracker.record(r.accuracy, r.confidence, r.latency,
                                category=item["category"], mode=mode.value)
        snap = self.tracker.snapshot()
        print(f"[1] Eval      : acc={snap['accuracy']*100:.1f}% conf={snap['confidence']*100:.1f}% p95={snap.get('latency_p95', 0):.2f}s")

        # Step 2: Feedback
        fb_agg = self.feedback.aggregate()
        fb_trigger, fb_reason = self.feedback.should_trigger_reeval()
        print(f"[2] Feedback  : avg_rating={fb_agg.get('avg_rating', 'N/A')} helpful={fb_agg.get('helpful_rate', 'N/A')} re_eval={fb_trigger}")

        # Step 3: Context freshness
        stale = [aid for aid, a in self.scheduler.assets.items() if a.is_stale]
        print(f"[3] Freshness : {len(stale)} stale asset(s) → {stale}")

        # Step 4: Alerts
        alerts = self.alert_eng.evaluate(snap)
        print(f"[4] Alerts    : {len(alerts)} fired")

        # Step 5: Tune
        actions_taken = []
        if alerts or fb_trigger:
            new_cfg = self.tuner.tune(alerts, snap)
            self.config = new_cfg
            actions_taken.append(f"config tuned to v{new_cfg.version}")

        # Step 6: Refresh stale assets
        refreshed = self.scheduler.run_refresh_cycle()
        if refreshed:
            actions_taken.append(f"refreshed assets: {refreshed}")

        print(f"[5] Actions   : {actions_taken if actions_taken else ['none — all metrics healthy']}")

        entry = {
            "cycle":         self.cycle_num,
            "mode":          mode.value,
            "accuracy":      snap["accuracy"],
            "confidence":    snap["confidence"],
            "alerts_fired":  len(alerts),
            "fb_trigger":    fb_trigger,
            "stale_assets":  len(stale),
            "actions":       actions_taken,
        }
        self.cycle_log.append(entry)
        return entry

    def final_report(self):
        print(f"\n{'='*60}\nFINAL PIPELINE REPORT — {self.cycle_num} cycles\n{'='*60}")
        for e in self.cycle_log:
            icon = "✅" if e["alerts_fired"] == 0 else "⚠️ "
            print(f"  {icon} Cycle {e['cycle']} [{e['mode']:8s}] acc={e['accuracy']*100:.1f}% "
                  f"alerts={e['alerts_fired']} actions={len(e['actions'])}")

# ── Simulate 3 eval cycles across degradation lifecycle ───────────────────────
pipeline = ContinuousEvalPipeline(
    initial_config=AgentConfig(system_prompt=SYSTEM_PROMPTS[AgentMode.HEALTHY])
)

NOW = datetime.now(timezone.utc)
pipeline.add_context_asset(ContextAsset("sp_main", "system_prompt", "initial prompt",
                                         NOW - timedelta(hours=180), ttl_hours=168))
pipeline.add_context_asset(ContextAsset("doc_faq", "retrieval_doc",  "old faq",
                                         NOW - timedelta(hours=10),  ttl_hours=72))

for signal in simulated_feedback:
    pipeline.add_feedback(signal)

print("=" * 60)
print("FULL PIPELINE — 3 cycles (HEALTHY → DRIFTED → DEGRADED)")
print("=" * 60)

pipeline.run_cycle(AgentMode.HEALTHY)
pipeline.run_cycle(AgentMode.DRIFTED)
pipeline.run_cycle(AgentMode.DEGRADED)

pipeline.final_report()


FULL PIPELINE — 3 cycles (HEALTHY → DRIFTED → DEGRADED)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
EVAL CYCLE 1 | mode=healthy | 19:52:41 UTC
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[1] Eval      : acc=90.0% conf=77.0% p95=3.62s
[2] Feedback  : avg_rating=2.71 helpful=0.429 re_eval=True
[3] Freshness : 1 stale asset(s) → ['sp_main']
  ⚠️ [WARNING ] latency_p95: 3.620 (threshold 3.0) — p95 latency 3.62s — approaching SLA limit.
[4] Alerts    : 1 fired
[5] Actions   : ['config tuned to v2', "refreshed assets: ['sp_main']"]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
EVAL CYCLE 2 | mode=drifted | 19:52:53 UTC
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
[1] Eval      : acc=85.0% conf=88.5% p95=3.62s
[2] Feedback  : avg_rating=2.71 helpful=0.429 re_eval=True
[3] Freshness : 0 stale asset(s) → []
  ⚠️ [WARNING ] latency_p95: 3.620 (threshold 3.0) — p95 latency 3.62s — approaching SLA limit.
[4] Alerts    : 1 fired
[5]